# Measure classification accuracy / F1 against gold labels

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/f-inverse/jammi-ai/blob/py-v0.49.1/cookbook/notebooks/recipes/eval_inference.ipynb)

Built from [`cookbook/recipes/eval_inference/example.py`](https://github.com/f-inverse/jammi-ai/blob/main/cookbook/recipes/eval_inference/example.py). Run the setup
cell first; every other cell runs top to bottom.

In [ ]:
# Setup: jammi 0.49.1 — the CUDA engine on an sm_80+ GPU (L4, A100, …), the
# CPU engine otherwise — and the cookbook's library and fixtures. On that GPU the
# chapter runs at `full` scale, over the published data; set SCALE = "small" to
# run the seconds-long version over the committed fixtures instead.
import os
import subprocess
import sys


def compute_capability() -> float:
    try:
        out = subprocess.run(
            ["nvidia-smi", "--query-gpu=compute_cap", "--format=csv,noheader"],
            capture_output=True, text=True, check=True,
        ).stdout.split()
    except (OSError, subprocess.CalledProcessError):
        return 0.0
    return float(out[0]) if out else 0.0


gpu = compute_capability() >= 8.0
engine = "jammi-ai-native-cu12" if gpu else "jammi-ai-native"
server = "jammi-server-cu12" if gpu else "jammi-server"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "jammi-ai==0.49.1", engine + "==0.49.1", "jammi-cookbook==0.49.1"], check=True)
SCALE = "full" if gpu else "small"
os.environ["JAMMI_COOKBOOK_SCALE"] = SCALE
print(f"engine: {engine}   scale: {SCALE}")

Run with `python cookbook/recipes/eval_inference/example.py`. Exits 0 on
success.

In [ ]:
from __future__ import annotations

import tempfile
from pathlib import Path

import jammi
from jammi_cookbook import fixtures

CORPUS_PATH = fixtures.path("tiny_corpus.parquet")
LABELS_PATH = fixtures.path("tiny_labels.csv")
MODEL = fixtures.model("tiny_modernbert_classifier")


def main() -> int:
    with tempfile.TemporaryDirectory() as tmp, jammi.connect(f"file://{tmp}") as db:

        # 1. Register the corpus and the gold labels.
        db.add_source("corpus", url=str(CORPUS_PATH), format="parquet")
        db.add_source("golden", url=str(LABELS_PATH), format="csv")

        # 2. Run inference + eval against the gold labels.
        metrics = db.eval_inference(
            model=MODEL,
            source="corpus",
            columns=["content"],
            task="classification",
            golden_source="golden.public.tiny_labels",
            label_column="label",
        )

        # 3. Sanity-check the aggregate metrics. `f1` is macro F1 averaged
        #    across classes. The aggregate is tagged by task kind.
        aggregate = metrics["aggregate"]
        assert aggregate["task"] == "classification", aggregate["task"]
        for key in ("accuracy", "f1"):
            value = aggregate[key]
            assert 0.0 <= value <= 1.0, f"{key} out of range: {value}"

        # 4. Per-class metrics live under `aggregate.per_class` — dict keyed
        #    by label.
        per_class = aggregate.get("per_class", {})
        assert isinstance(per_class, dict), f"per_class shape: {type(per_class)}"

        # 5. Per-record predictions live under `per_record` (one entry per
        #    aligned predicted/gold pair).
        per_record = metrics["per_record"]
        assert len(per_record) > 0, "per_record must carry one entry per aligned row"

        print(f"accuracy:  {aggregate['accuracy']:.4f}")
        print(f"macro_f1:  {aggregate['f1']:.4f}")
        print("per_class:")
        for label, stats in per_class.items():
            print(
                f"  {label:<12} precision={stats['precision']:.4f}"
                f"  recall={stats['recall']:.4f}  f1={stats['f1']:.4f}"
            )
        print(f"per_record: {len(per_record)} predictions")

        # 6. The predictions themselves, without the gold labels: `infer` runs
        #    the model over the source and returns one row per source row,
        #    keyed by `id`, with the model's outputs as columns.
        predictions = db.infer(
            source="corpus",
            model=MODEL,
            columns=["content"],
            task="classification",
            key="id",
        )
        assert predictions.num_rows == 20, predictions.num_rows
        print(f"infer: {predictions.num_rows} rows, columns {predictions.column_names}")

    print("eval_inference: OK")
    return 0

In [ ]:
assert main() == 0